# 2단계 · 데이터 분할 (train / validation / test)

**이 노트북이 하는 일:** 전체 939개 데이터를 세 묶음으로 나눕니다.

| 묶음 | 비율 | 역할 |
|------|------|------|
| 학습 (train) | 70% | 모델이 공부하는 문제집 |
| 검증 (validation) | 15% | 학습 중 중간점검용 모의고사 |
| 테스트 (test) | 15% | 맨 마지막에 딱 한 번 보는 실전 시험 |

**핵심 원칙 2가지**
1. **데이터 누수 방지** — 테스트셋은 학습에 절대 쓰지 않습니다. (공부한 문제로 시험 보면 실력 측정이 안 되니까요)
2. **계층적 분할(stratified)** — 세 묶음 모두에서 of/from/with 비율을 똑같이 유지합니다. (데이터가 불균형하기 때문)

In [1]:
# pandas: CSV 같은 표 데이터를 읽고 다루는 도구. 관례적으로 'pd'라는 짧은 별칭으로 부릅니다.
import pandas as pd

# scikit-learn의 train_test_split: 데이터를 무작위로 두 묶음으로 나눠주는 함수입니다.
from sklearn.model_selection import train_test_split

# pathlib.Path: 파일/폴더 경로를 안전하게 다루는 표준 도구입니다.
from pathlib import Path

In [2]:
# CSV 파일의 위치. 이 노트북은 notebooks/ 폴더 안에 있으므로,
# '..'(상위 폴더)로 올라가 data/made_prepositions.csv 를 가리킵니다.
csv_path = Path("..") / "data" / "made_prepositions.csv"

# CSV 파일을 표(DataFrame) 형태로 읽어옵니다. df는 'data frame'의 약자입니다.
df = pd.read_csv(csv_path)

# 전체 행 수를 출력해 제대로 읽혔는지 확인합니다(939개가 나와야 합니다).
print("전체 행 수:", len(df))

# 처음 5줄을 표로 보여줍니다. 어떤 열들이 있는지 눈으로 확인하는 용도입니다.
df.head()

전체 행 수: 939


,id,label,product,material,sentence,sentence_masked
0,1,of,Each component,many component units,Each component in turn is made of many compone...,Each component in turn is made ___ many compon...
1,2,of,a corporation,natural persons who are citizens,"In Deveaux , Marshall admits that a corporatio...","In Deveaux , Marshall admits that a corporatio..."
2,3,of,The apparatus,wood,"The apparatus was made of wood , and consisted...","The apparatus was made ___ wood , and consiste..."
3,4,of,all walls more than three feet tall,stone,Apparently all walls more than three feet tall...,Apparently all walls more than three feet tall...
4,5,of,a portrait,the white stepdaughter,the white stepdaughter is “ duplicated ” when ...,the white stepdaughter is “ duplicated ” when ...


In [3]:
# label(정답 전치사) 열에서 of/from/with가 각각 몇 개인지 셉니다.
print("개수:")
print(df["label"].value_counts())

# 같은 내용을 비율(%)로도 봅니다. normalize=True가 '개수 대신 비율'로 바꿔줍니다.
print("\n비율(%):")
print((df["label"].value_counts(normalize=True) * 100).round(1))

개수:
label
of      511
from    248
with    180
Name: count, dtype: int64

비율(%):
label
of      54.4
from    26.4
with    19.2
Name: proportion, dtype: float64


### 왜 두 번에 나눠 자르나요?

`train_test_split`은 한 번에 **두 묶음**만 만들 수 있습니다. 우리는 **세 묶음**이 필요하므로 두 단계로 자릅니다:

1. 먼저 전체를 **(학습+검증) 85% / 테스트 15%** 로 자른다.
2. 남은 85%를 다시 **학습 / 검증** 으로 자른다.

- `stratify=...` : 자를 때 of/from/with 비율을 그대로 유지 (불균형 대응).
- `random_state=42` : 무작위로 섞되 **항상 같은 결과**가 나오도록 고정 (재현성 — 연구에서 매우 중요).

In [4]:
# [1단계 자르기] 전체(df)를 (학습+검증) 묶음과 테스트 묶음으로 나눕니다.
#   test_size=0.15      -> 전체의 15%를 테스트로 떼어냄
#   stratify=df["label"]-> 세 전치사 비율을 양쪽 묶음에서 동일하게 유지
#   random_state=42     -> 같은 결과를 재현하기 위한 고정 시드
train_val_df, test_df = train_test_split(
    df,
    test_size=0.15,
    stratify=df["label"],
    random_state=42,
)

# [2단계 자르기] 남은 85%(train_val_df)를 학습과 검증으로 나눕니다.
#   전체 기준 15%를 검증으로 떼려면, 85% 안에서는 0.15/0.85 = 약 0.1765 비율을 떼야 합니다.
val_ratio = 0.15 / 0.85
train_df, val_df = train_test_split(
    train_val_df,
    test_size=val_ratio,
    stratify=train_val_df["label"],
    random_state=42,
)

# 세 묶음의 크기를 출력합니다.
print("학습(train):", len(train_df))
print("검증(val)  :", len(val_df))
print("테스트(test):", len(test_df))

학습(train): 657
검증(val)  : 141
테스트(test): 141


In [5]:
# [검증] 세 묶음 모두에서 of/from/with 비율이 전체와 비슷하게 유지됐는지 확인합니다.
# 비율이 묶음마다 들쭉날쭉하면 stratify가 제대로 안 된 것이므로 꼭 점검합니다.
for name, part in [("전체", df), ("학습", train_df), ("검증", val_df), ("테스트", test_df)]:
    # 각 묶음에서 전치사별 비율(%)을 계산해 보기 좋게 정리합니다.
    pct = (part["label"].value_counts(normalize=True) * 100).round(1).to_dict()
    print(f"{name:3s} (n={len(part):3d}): {pct}")

전체  (n=939): {'of': 54.4, 'from': 26.4, 'with': 19.2}
학습  (n=657): {'of': 54.3, 'from': 26.5, 'with': 19.2}
검증  (n=141): {'of': 54.6, 'from': 26.2, 'with': 19.1}
테스트 (n=141): {'of': 54.6, 'from': 26.2, 'with': 19.1}


In [6]:
# 나눈 세 묶음을 파일로 저장합니다. 다음 단계(베이스라인/파인튜닝)에서 이 파일들을 불러 씁니다.
# data/splits/ 폴더에 저장하며, 폴더가 없으면 자동으로 만듭니다.
out_dir = Path("..") / "data" / "splits"
out_dir.mkdir(parents=True, exist_ok=True)  # parents=True: 중간 폴더도 함께 생성

# index=False -> pandas가 자동으로 붙이는 행 번호 열을 파일에 쓰지 않습니다(불필요하므로).
train_df.to_csv(out_dir / "train.csv", index=False)
val_df.to_csv(out_dir / "val.csv", index=False)
test_df.to_csv(out_dir / "test.csv", index=False)

# 저장 위치를 절대경로로 출력해 어디에 만들어졌는지 확인합니다.
print("저장 완료 ->", out_dir.resolve())

저장 완료 -> /Users/jamie/Desktop/00_github_repositories/myproject/data/splits
